# 00 Daten

Kern-Daten (Chip, Emulator, FP32) laden, prüfen und zu Tabellen je Bild aufbereiten. Alle weiteren Notebooks lesen nur diese Tabellen.
Plan: Vault-Notiz „Auswertungsplan“ (Tag `auswertungsplan-v1`). Logik: `thesis.analysis.kern`, `thesis.analysis.maps`.

In [ ]:
import numpy as np
import pandas as pd
from thesis.analysis import kern

N_JOBS = 4                     # parallele Prozesse, je Prozess bis ≈ 3 GB RAM
SPEC, CACHE = kern.KernSpec(), kern.CACHE_DIR
kern.provenance()

## 1 Inventar

**Was:** alle Run-Ordner des Kerns aus `results/*/meta.json` einlesen.
**Warum:** Grundlage für Prüfung und Extraktion; zeigt auf einen Blick, ob jede Quelle in voller Blockzahl da ist.

In [ ]:
inv = kern.inventory(SPEC)
inv.pivot_table(index="source", columns="model", values="n", aggfunc=["count", "sum"])

## 2 Prüfung

**Was:** Blöcke lückenlos, alle Dateien da, Bild-IDs = Liste, kein Lauf `dirty`, HEF-Hash je Compile gleich (Chip = Emulator), Emulator-Zielklassen aus dem richtigen FP32-Lauf.
**Warum:** Läufe mit Fehlern dürfen nicht in die Auswertung (Plan Abschn. 8).

In [ ]:
checks = kern.check_inventory(inv, SPEC)
print(f"{checks.ok.sum()}/{len(checks)} Prüfungen bestanden")
checks[~checks.ok]

In [ ]:
assert checks.ok.all(), "Prüfung fehlgeschlagen – nicht weitermachen"

## 3 Extraktion

**Was:** jeden Block einmal lesen und speichern: 7×7-Rohkarten (Eigen-CAM, Grad-CAM), Ausgabegrößen je Bild (Top-1/5, Logit-Cosinus, ΔP-Basis, Ziel-Logit) und den Aktivierungsabstand Emulator ↔ Chip (TF2).
**Warum:** Die Rohdaten sind ≈ 36 GB; danach arbeiten alle Notebooks mit wenigen hundert MB. Wiederaufnehmbar: fertige Blöcke werden übersprungen.

In [ ]:
%%time
kern.extract_all(inv, SPEC, CACHE, n_jobs=N_JOBS)

## 4 Konsistenz

**Was:** (a) FP32 auf der Teilmenge (eigener Lauf) gegen FP32 auf allen 48 000 für dieselben Bilder; (b) Emulator k01 auf der Teilmenge gegen den vollen Emulator-Lauf k01.
**Warum:** Dieselben Bilder wurden zweimal gerechnet. Erwartet: gleich oder nur Gleitkomma-Rauschen (anderer Batch). Sonst ist unklar, welche Fassung gilt.

In [ ]:
def gleich(model, name_a, name_b):
    A, B = kern.load_source(CACHE, model, name_a), kern.load_source(CACHE, model, name_b)
    pos = {i: n for n, i in enumerate(A["ids"])}
    ia = np.array([pos[i] for i in B["ids"]])
    zeile = {"model": model, "vergleich": f"{name_a} ↔ {name_b}", "n": len(ia)}
    for key in ("eig_raw", "grad_raw", "logits", "logit_t"):
        if key in A and key in B:
            zeile[f"max|Δ| {key}"] = float(np.abs(A[key][ia] - B[key]).max())
    if "top1" in A and "top1" in B:
        zeile["Top-1 gleich"] = float((A["top1"][ia] == B["top1"]).mean())
    if "logits" in A and "logits" in B:
        zeile["Top-1 gleich"] = float((A["logits"][ia].argmax(1) == B["logits"].argmax(1)).mean())
    return zeile

pd.DataFrame([gleich(m, "fp32", "fp32_teil") for m in SPEC.models]
             + [gleich(m, "emuvoll_k01", "emu_k01") for m in SPEC.models])

## 5 Tabellen

**Was:** je Modell drei Tabellen in `auswertung_cache/tabellen/`:
`quellen` (Quelle × Bild: Vorhersage, Ausgabegrößen, Sättigung, leere/konstante Karte),
`vergleiche` (Paar × Bild: Top-15/20-IoU, CC, ausgerichtet `*_al`, Spearman/KLD bei FP32 ↔ X),
`aktivierung` (Emulator ↔ Chip je Bild: relativer Fehler, Cosinus, Anteil bitgleich).
**Warum:** Grundlage für 01–06. Die Definitionen folgen dem Plan (Abschn. 3–5, 8): Eigen-CAM ohne FP32-Ausrichtung, Grad-CAM-Metriken leer (NaN), wenn eine Karte leer ist.

In [ ]:
%%time
pfade = kern.compute_tables(SPEC, CACHE, n_jobs=N_JOBS)
pfade

In [ ]:
for model in SPEC.models:
    q = pd.read_parquet(pfade[f"quellen_{model}"])
    v = pd.read_parquet(pfade[f"vergleiche_{model}"])
    a = pd.read_parquet(pfade[f"aktivierung_{model}"])
    falsch = int((q.grad_target_ok == False).sum())          # Grad-CAM-Ziel ≠ FP32-Top-1, erwartet 0
    print(f"{model}: quellen {len(q):,} · vergleiche {len(v):,} · aktivierung {len(a):,} · Grad-CAM-Ziel falsch: {falsch}")
    display(v.assign(b_art=v.b.astype(str).str.split("_").str[0])
             .groupby(["cam", "a", "b_art"], observed=True).image_id.agg(["size", "nunique"]))

## 6 Plausibilität FP32

**Was:** FP32-Top-1/Top-5 auf den 48 000 Bildern.
**Warum:** bestätigt Labels und Bildzuordnung. Soll (Logbuch 17.11.2026): ResNet18 69,89 %, MobileNetV3 74,08 %; torchvision auf 50 000: 69,758 % bzw. 74,042 %. Hier nur FP32, keine INT8-Ergebnisse.

In [ ]:
pd.concat([pd.read_parquet(pfade[f"quellen_{m}"]).query("source == 'fp32'") for m in SPEC.models]) \
  .groupby("model")[["top1_ok", "top5_ok"]].mean().mul(100).round(2)